## ▶ What you'll see when you run this
- A tricky word problem the model gets **wrong** with a naive prompt and **right** once we add "think step by step" — plus a quick majority vote.

**Time:** ~8 min · **Cost:** free (cheapest model: Gemini Flash / Claude Haiku) · **Keys:** none required (scripted fallback runs the whole demo) — add `GEMINI_API_KEY` or `ANTHROPIC_API_KEY` for the real thing.

# Week 9 · Notebook 3 — Reasoning & Chain-of-Thought
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Some problems need the model to **work through steps** before answering. This notebook shows:
1. A problem a model gets **wrong** with a naive prompt, **right** with CoT.
2. **Zero-shot CoT** ("think step by step") vs **few-shot CoT** (worked examples).
3. **Self-consistency**: sample a few CoT answers, take the **majority vote**.

Compared across **Claude AND Gemini**. Degrades gracefully with **no key**.

## 0. Quick visible result (no key needed)
Before any API calls, here is the puzzle and the *parser* we'll reuse. Run this to see the question and the right answer up front.

In [ ]:
import re

PUZZLE = (
  'A bat and a ball cost $1.10 in total. '
  'The bat costs $1.00 more than the ball. '
  'How much does the ball cost (in cents)?'
)
CORRECT = '5'   # ball = $0.05; the tempting wrong answer is 10 cents

def final_number(text):
    """Pull the last number out of a model answer (handles 'Answer: 5')."""
    nums = re.findall(r'-?\d+(?:\.\d+)?', text or '')
    if not nums:
        return ''
    n = nums[-1]
    return n[:-2] if n.endswith('.0') else n   # 5.0 -> 5, but keep 10

print('PUZZLE:', PUZZLE)
print('Correct answer (cents):', CORRECT)
print('parser self-test ->', final_number('... so Answer: 5 cents'))

## 1. Install + load keys safely

In [ ]:
!pip -q install anthropic google-generativeai

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ('ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except Exception:
    pass
HAVE_CLAUDE = bool(os.environ.get('ANTHROPIC_API_KEY'))
HAVE_GEMINI = bool(os.environ.get('GEMINI_API_KEY'))
print('Claude key:', HAVE_CLAUDE, '| Gemini key:', HAVE_GEMINI)

## 2. One ask() across providers — with a scripted fallback
`ask()` calls Claude or Gemini if a key is present, otherwise returns a *canned* response so the whole reasoning demo still runs. The fallback deliberately answers **'10'** to the naive prompt and **'5'** when it sees 'step by step' — exactly the effect CoT has on real models.

In [ ]:
def _ask_claude(prompt, temperature=0.0, max_tokens=400):
    import anthropic
    c = anthropic.Anthropic()
    m = c.messages.create(model='claude-haiku-4-5-20251001',
        max_tokens=max_tokens, temperature=temperature,
        messages=[{'role': 'user', 'content': prompt}])
    return m.content[0].text

def _ask_gemini(prompt, temperature=0.0, max_tokens=400):
    import google.generativeai as genai
    genai.configure(api_key=os.environ['GEMINI_API_KEY'])
    r = genai.GenerativeModel('gemini-2.5-flash').generate_content(
        prompt, generation_config={'temperature': temperature,
                                   'max_output_tokens': max_tokens})
    return r.text

def _ask_fallback(prompt, temperature=0.0, max_tokens=400):
    """No-key stand-in: mimics the naive-vs-CoT behavior + sampling noise."""
    import random
    if 'step by step' in prompt.lower() or 'reason' in prompt.lower():
        # With reasoning, mostly right (5) but occasionally slips (self-consistency).
        val = random.choices(['5', '10'], weights=[0.8, 0.2])[0]
        return f'Ball + bat = 110; bat = ball + 100; 2*ball = 10. Answer: {val}'
    return 'It looks like 10 cents. Answer: 10'   # classic naive miss

def ask(prompt, provider='auto', temperature=0.0, max_tokens=400):
    if provider in ('auto', 'gemini') and HAVE_GEMINI:
        return _ask_gemini(prompt, temperature, max_tokens)
    if provider in ('auto', 'claude') and HAVE_CLAUDE:
        return _ask_claude(prompt, temperature, max_tokens)
    return _ask_fallback(prompt, temperature, max_tokens)

print('ask() ready — using',
      'Gemini' if HAVE_GEMINI else 'Claude' if HAVE_CLAUDE else 'scripted fallback')

## 3. Naive prompt (often wrong) vs "think step by step" (right)
Same puzzle, two prompts. The naive one rushes to the tempting answer; the CoT one forces the model to lay out the algebra first.

In [ ]:
naive = PUZZLE + '\nAnswer with just the number of cents.'
cot   = PUZZLE + '\nThink step by step, then end with a line: Answer: <cents>'

a_naive = ask(naive)
a_cot   = ask(cot, max_tokens=300)
print('--- NAIVE ---'); print(a_naive)
print('  parsed:', final_number(a_naive),
      '->', 'CORRECT' if final_number(a_naive) == CORRECT else 'WRONG')
print('\n--- CHAIN-OF-THOUGHT ---'); print(a_cot)
print('  parsed:', final_number(a_cot),
      '->', 'CORRECT' if final_number(a_cot) == CORRECT else 'WRONG')

## 4. Zero-shot CoT vs few-shot CoT
**Zero-shot CoT** = just say *think step by step*. **Few-shot CoT** = also show one or two **worked examples** of the reasoning style. Few-shot teaches the *format* of good reasoning, which helps on harder or unusual problems.

In [ ]:
zero_shot_cot = PUZZLE + '\nThink step by step, then: Answer: <cents>'

few_shot_cot = (
  'Solve word problems by reasoning step by step, then a final Answer line.\n\n'
  'Q: Two pencils cost $0.30 total; one costs $0.20 more than the other. '
  'Cheaper one in cents?\n'
  'Reasoning: cheap + (cheap+20) = 30 -> 2*cheap = 10 -> cheap = 5.\n'
  'Answer: 5\n\n'
  f'Q: {PUZZLE}\nReasoning:'
)

for label, p in [('ZERO-SHOT CoT', zero_shot_cot), ('FEW-SHOT CoT', few_shot_cot)]:
    out = ask(p, max_tokens=300)
    print(f'=== {label} ===')
    print(out)
    print('  parsed:', final_number(out),
          '->', 'CORRECT' if final_number(out) == CORRECT else 'WRONG', '\n')

## 5. Self-consistency: sample a few, take the majority vote
Reasoning at `temperature>0` makes the model explore different paths. **Self-consistency** runs the CoT prompt several times and **votes** on the final answer — a cheap accuracy boost over a single sample.

In [ ]:
from collections import Counter

def self_consistency(prompt, n=5, temperature=0.7):
    answers = [final_number(ask(prompt, temperature=temperature, max_tokens=300))
               for _ in range(n)]
    winner, votes = Counter(answers).most_common(1)[0]
    return winner, answers

winner, samples = self_consistency(cot, n=5)
print('samples:', samples)
print('majority vote ->', winner,
      '|', 'CORRECT' if winner == CORRECT else 'WRONG')
print('(One unlucky sample can be wrong; the vote is more robust.)')

## 6. Claude vs Gemini on the same reasoning prompt
If you have **both** keys, compare how each model reasons. The technique is identical; only the call surface differs (see Notebook 2).

In [ ]:
for prov in ('claude', 'gemini'):
    have = HAVE_CLAUDE if prov == 'claude' else HAVE_GEMINI
    if not have:
        print(f'[{prov}] no key — skipping'); continue
    out = ask(cot, provider=prov, max_tokens=300)
    print(f'=== {prov.upper()} ===')
    print(out)
    print('  parsed:', final_number(out), '\n')

## 7. Extended thinking / test-time compute (2026)
CoT is *you* asking for steps. **Extended thinking** lets the model spend a **budget** of hidden reasoning tokens *before* it answers — the "think longer" knob. **Claude** takes `thinking={'type':'enabled','budget_tokens':N}`; **Gemini 2.5** has a `thinking` mode. Great for hard, multi-step problems; on easy ones it just costs more ("overthinking"). Open reasoning models like **DeepSeek-R1** do this by default (run locally via Ollama). Runs offline below via a scripted fallback.

In [ ]:
def think_claude(prompt, budget_tokens=1024, max_tokens=2000):
    """Claude with extended thinking enabled (needs ANTHROPIC_API_KEY)."""
    import anthropic
    c = anthropic.Anthropic()
    m = c.messages.create(
        model='claude-sonnet-4-6', max_tokens=max_tokens,
        thinking={'type': 'enabled', 'budget_tokens': budget_tokens},
        messages=[{'role': 'user', 'content': prompt}])
    # response has thinking block(s) then the final text block
    think = ''.join(b.thinking for b in m.content if b.type == 'thinking')
    answer = ''.join(b.text for b in m.content if b.type == 'text')
    return think, answer

def think_gemini(prompt, budget_tokens=1024, max_tokens=2000):
    """Gemini 2.5 'thinking' variant (needs GEMINI_API_KEY)."""
    import google.generativeai as genai
    genai.configure(api_key=os.environ['GEMINI_API_KEY'])
    r = genai.GenerativeModel('gemini-2.5-flash').generate_content(
        prompt, generation_config={'max_output_tokens': max_tokens,
                                   'thinking_config': {'thinking_budget': budget_tokens}})
    return '(thinking hidden by API)', r.text

def think_fallback(prompt, budget_tokens=1024, max_tokens=2000):
    """No-key stand-in: shows a budgeted scratchpad, then the right answer."""
    scratch = ('Let ball = b cents. bat = b + 100. b + (b+100) = 110 -> '
               '2b = 10 -> b = 5. (used ~60 of %d budget tokens)' % budget_tokens)
    return scratch, 'Answer: 5'

prompt = PUZZLE + '\nReason carefully, then end with: Answer: <cents>'
if HAVE_CLAUDE:    thinking, answer = think_claude(prompt)
elif HAVE_GEMINI:  thinking, answer = think_gemini(prompt)
else:              thinking, answer = think_fallback(prompt)

print('--- THINKING (hidden scratchpad) ---'); print(thinking)
print('\n--- ANSWER ---'); print(answer)
print('  parsed:', final_number(answer),
      '->', 'CORRECT' if final_number(answer) == CORRECT else 'WRONG')
print('\nTip: raise/lower budget_tokens to trade accuracy for cost; on EASY'
      ' tasks a big budget can cause "overthinking" (diminishing returns).')

---
### Takeaways
- **Naive → CoT** can flip a wrong answer to a right one with *zero* model change.
- **Few-shot CoT** teaches the reasoning *format*; helps on harder problems.
- **Self-consistency** (sample + vote) beats a single sample — at extra cost.
- **Extended thinking / test-time compute** lets the model think longer via a token **budget** — worth it for hard problems, wasteful ("overthinking") on easy ones.
- Reasoning is for **multi-step** work (math, logic, planning); simple lookups don't need it.

### Tasks (ungraded practice — feeds Capstone M1)
1. Swap in your own multi-step problem; show naive-wrong vs CoT-right.
2. Try `n=3` vs `n=7` in self-consistency — does the vote stabilize?
3. If you have both keys, note one difference in how Claude vs Gemini reason.

In [ ]:
# your reasoning experiments here
